# 🎴 Pokémon TCG RL — Live Training Notebook

Trains the `PolicyNetwork` against the real **cabt** engine on Kaggle's Linux runtime.

## Before running
1. **Accelerator** → Settings → GPU (T4 x2 or P100)
2. **Internet** → Settings → On
3. **Add datasets**:
   - `pokemage-src` (your source code dataset) — required
   - `pokemage-weights` (previous checkpoint) — optional, for resuming

## Session time
Kaggle notebooks allow up to **9 hours** per session.  
The batched runner in Cell 6 respects this limit automatically.


---
## Cell 1 — Environment check

In [ ]:
import os, sys, platform, subprocess
import torch

print(f"  OS          : {platform.system()} {platform.release()}")
print(f"  Python      : {sys.version.split()[0]}")
print(f"  Working dir : {os.getcwd()}")
print(f"  PyTorch     : {torch.__version__}")
print(f"  CUDA Avail  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"  CUDA Version: {torch.version.cuda}")
    print(f"  GPU Device  : {torch.cuda.get_device_name(0)}")
    try:
        x = torch.randn(2, 2, device="cuda")
        print("  CUDA Test   : PASSED ✓")
    except Exception as e:
        print(f"  CUDA Test   : FAILED ✗ ({e})")
        print("\n  [!] Troubleshooting: If you get 'no kernel image is available', do one of:\n"
              "      1. Click 'Run' -> 'Factory Reset Solo Session' to restore default PyTorch.\n"
              "      2. Switch the GPU Accelerator type in the right sidebar (e.g., T4 x2 vs P100).")
else:
    gpu_info = subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo 'no GPU'")
    print(f"  GPU         : {gpu_info.strip()}")

# Check CPU cores
ncpu = os.cpu_count()
print(f"  CPU cores   : {ncpu}")

# Check available disk space (GB)
disk = subprocess.getoutput("df -BG /kaggle/working | tail -1 | awk '{print $4}'")
print(f"  Disk free   : {disk}")

print("\n  ✓ Environment check complete")

---
## Cell 2 — Install dependencies

In [ ]:
%%bash
# kaggle-environments is not preinstalled on Kaggle
pip install -q 'kaggle-environments>=1.14.0'

echo "Dependencies installed ✓"

---
## Cell 3 — Mount source code dataset

In [ ]:
import shutil, os, sys

# ── Locate the source dataset ────────────────────────────────────────────────
# Kaggle mounts datasets under /kaggle/input/<dataset-slug>/
# Adjust the slug below if you named your dataset differently.
SRC_CANDIDATES = [
    "/kaggle/input/pokemage-src",
    "/kaggle/input/pokemagesrc",
    "/kaggle/input/pokemage",
    "/kaggle/input/dollis/datasets/pokemage-src",
    "/kaggle/input/dollis/datasets/pokemage",
    "kaggle/dollis/datasets/pokemage-src",
    "kaggle/dollis/datasets/pokemage",
]
SRC = next((p for p in SRC_CANDIDATES if os.path.isdir(p)), None)
DST = "/kaggle/working"

if SRC is None:
    raise FileNotFoundError(
        "Source dataset not found. "
        f"Searched: {SRC_CANDIDATES}\n"
        "Add your 'pokemage-src' dataset as an input to this notebook."
    )

print(f"  Source dataset : {SRC}")

os.makedirs(os.path.join(DST, "src"), exist_ok=True)

# ── Copy source files ─────────────────────────────────────────────────────────
SOURCE_FILES = [
    "agent.py",
    "config.py",
    "card_data.py",
    "env_wrapper.py",
    "model.py",
    "train.py",
    "eval.py",
    "smoke_test.py",
]

for f in SOURCE_FILES:
    src_path = os.path.join(SRC, f)
    dst_path = os.path.join(DST, "src", f)
    if os.path.exists(src_path):
        shutil.copy(src_path, dst_path)
        print(f"  ✓  src/{f}")
    else:
        print(f"  ⚠  src/{f} not found in source dataset (skipping)")

# Copy requirements.txt
req_src = os.path.join(SRC, "requirements.txt")
req_dst = os.path.join(DST, "requirements.txt")
if os.path.exists(req_src):
    shutil.copy(req_src, req_dst)
    print("  ✓  requirements.txt")
else:
    print("  ⚠  requirements.txt not found in source dataset (skipping)")

# ── Copy EN_Card_Data.csv if present ─────────────────────────────────────────
csv_src = os.path.join(SRC, "EN_Card_Data.csv")
csv_dst = os.path.join(DST, "EN_Card_Data.csv")
if os.path.exists(csv_src):
    shutil.copy(csv_src, csv_dst)
    print(f"  ✓  EN_Card_Data.csv  ({os.path.getsize(csv_dst)//1024} KB)")
else:
    print("  ⚠  EN_Card_Data.csv not found – agent will use synthetic vocab")

# ── Add working dir to Python path ───────────────────────────────────────────
if DST not in sys.path:
    sys.path.insert(0, DST)
src_dir = os.path.join(DST, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print(f"\n  ✓ All files ready in {DST}")

---
## Cell 4 — Resume from previous checkpoint (optional)

In [ ]:
CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# Search for an existing checkpoint to resume from
CKPT_SOURCES = [
    "/kaggle/input/pokemage-weights/latest.pth",
    "/kaggle/input/pokemageweights/latest.pth",
    "/kaggle/input/dollis/datasets/pokemage-weights/latest.pth",
    "/kaggle/input/dollis/datasets/pokemageweights/latest.pth",
    "kaggle/dollis/datasets/pokemage-weights/latest.pth",
    "kaggle/dollis/datasets/pokemageweights/latest.pth",
    f"{CKPT_DIR}/latest.pth",   # already in working dir from a previous run
]

RESUME_CKPT = None
for p in CKPT_SOURCES:
    if os.path.exists(p):
        RESUME_CKPT = p
        break

if RESUME_CKPT and RESUME_CKPT != f"{CKPT_DIR}/latest.pth":
    dst = f"{CKPT_DIR}/latest.pth"
    shutil.copy(RESUME_CKPT, dst)
    size_mb = os.path.getsize(dst) / 1e6
    print(f"  ✓ Checkpoint copied from {RESUME_CKPT}")
    print(f"    Size: {size_mb:.1f} MB")
    print(f"    Will resume training from this checkpoint.")
elif RESUME_CKPT:
    size_mb = os.path.getsize(RESUME_CKPT) / 1e6
    print(f"  ✓ Found existing checkpoint: {RESUME_CKPT}  ({size_mb:.1f} MB)")
else:
    print("  ℹ No checkpoint found – training will start from scratch.")
    print("    (Add 'pokemage-weights' dataset as input to resume from a previous run)")

---
## Cell 5 — Smoke test (verify pipeline before long training)

In [ ]:
%%bash
cd /kaggle/working
python src/smoke_test.py 2>&1 | tail -20

---
## Cell 6 — Training

Configure the parameters below, then run.  
The batched runner starts a **fresh subprocess per batch** to flush the `libcg.so` C++ memory leak.

**Time budget:**  
- Kaggle allows ~9 hours per session → ~32,400 seconds  
- Each game takes ~2–5 seconds on live cabt  
- 5,000 games × 3s ≈ 4 hours — safe for one session  
- Set `TOTAL_GAMES=10000` for a full overnight run (may hit time limit)


In [ ]:
# ── Training configuration ────────────────────────────────────────────────────
# Adjust these before running.

TOTAL_GAMES  = 5000        # Total games to train for this session
BATCH_SIZE   = 200         # Games per subprocess (controls memory flush frequency)
ALGO         = "ppo"       # "ppo" (stable, recommended) or "reinforce" (faster)
ENV          = "live"      # "live" (real cabt on Kaggle) or "mock" (for testing)
DECK         = "dragapult_ex"  # Starter deck
BATCH_GAMES  = 20          # Episodes per gradient update
LR           = 3e-4        # Adam learning rate
OUTDIR       = "/kaggle/working/checkpoints"

# Resume from checkpoint?
import glob
existing = f"{OUTDIR}/latest.pth"
RESUME   = existing if os.path.exists(existing) else ""

print("Training configuration")
print("─" * 40)
print(f"  Total games  : {TOTAL_GAMES:,}")
print(f"  Batch size   : {BATCH_SIZE} games/subprocess")
print(f"  Algorithm    : {ALGO}")
print(f"  Environment  : {ENV}")
print(f"  Deck         : {DECK}")
print(f"  Learning rate: {LR}")
print(f"  Checkpoint   : {RESUME or 'starting fresh'}")
print(f"  Output dir   : {OUTDIR}")
print("─" * 40)

In [ ]:
import subprocess, time

os.makedirs(OUTDIR, exist_ok=True)

games_done  = 0
batch_num   = 0
session_start = time.time()
SESSION_LIMIT_S = 8.5 * 3600   # stop 30min before Kaggle kills the session

print(f"Starting batched training — {TOTAL_GAMES:,} games in batches of {BATCH_SIZE}")
print(f"Session limit: {SESSION_LIMIT_S/3600:.1f} hours")
print()

while games_done < TOTAL_GAMES:
    elapsed_session = time.time() - session_start
    if elapsed_session > SESSION_LIMIT_S:
        print(f"\n  ⏱ Session limit reached ({elapsed_session/3600:.2f}h). Stopping safely.")
        break

    batch_num  += 1
    games_this  = min(BATCH_SIZE, TOTAL_GAMES - games_done)

    # Build checkpoint flag
    ckpt_flag = ""
    latest = f"{OUTDIR}/latest.pth"
    if os.path.exists(latest):
        ckpt_flag = f"--checkpoint {latest}"

    cmd = (
        f"cd /kaggle/working && python src/train.py "
        f"--num_games {games_this} "
        f"--algo {ALGO} "
        f"--env {ENV} "
        f"--deck {DECK} "
        f"--batch_games {BATCH_GAMES} "
        f"--lr {LR} "
        f"--outdir {OUTDIR} "
        f"--flush_every {games_this} "
        f"{ckpt_flag}"
    )

    t0 = time.time()
    print(f"Batch {batch_num:3d} | games {games_done+1:>6,} → {games_done+games_this:<6,} ", end="", flush=True)

    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    dt = time.time() - t0
    gps = games_this / max(dt, 1)

    if result.returncode == 0:
        print(f"| {dt:5.0f}s | {gps:.1f} g/s | ✓")
    else:
        print(f"| {dt:5.0f}s | ✗ FAILED")
        # Print last 20 lines of stderr for diagnosis
        err_lines = (result.stderr or "").strip().splitlines()
        for line in err_lines[-20:]:
            print(f"    {line}")

    games_done += games_this

total_time = time.time() - session_start
print()
print("─" * 60)
print(f"  Completed {games_done:,} games in {total_time/60:.1f} min")
print(f"  Average speed : {games_done/max(total_time,1):.2f} games/s")

# List checkpoints
ckpts = sorted(glob.glob(f"{OUTDIR}/*.pth"))
print(f"  Checkpoints saved : {len(ckpts)}")
if ckpts:
    latest_size = os.path.getsize(f"{OUTDIR}/latest.pth") / 1e6
    print(f"  latest.pth        : {latest_size:.1f} MB")

---
## Cell 7 — Quick evaluation (sanity check)

In [ ]:
%%bash
cd /kaggle/working
python src/eval.py \
    --checkpoint ./checkpoints/latest.pth \
    --num_episodes 5 \
    2>&1

---
## Cell 8 — Agent self-test

In [ ]:
%%bash
cd /kaggle/working
python src/agent.py --selftest 2>&1

---
## Cell 9 — Save checkpoint as Kaggle dataset output

After this notebook runs, Kaggle saves everything in `/kaggle/working/` as **Output files**.  
To persist the checkpoint between sessions:

1. Click **"Data"** tab → **"Output"** → find `checkpoints/latest.pth`  
2. Click **"Create Dataset from output"**  
3. Name it **`pokemage-weights`**  
4. Next session: add `pokemage-weights` as an input → Cell 4 will auto-resume

In [ ]:
import glob

print("Output files ready for download / dataset creation:")
print()

output_files = sorted(glob.glob("/kaggle/working/checkpoints/*.pth"))
if output_files:
    for f in output_files:
        size_mb = os.path.getsize(f) / 1e6
        print(f"  📦  {os.path.basename(f):35s}  {size_mb:.1f} MB")
else:
    print("  ⚠ No checkpoints found — did training run successfully in Cell 6?")

print()
print("Next steps:")
print("  1. Data tab → Output → Create Dataset → name it 'pokemage-weights'")
print("  2. Run make_submission.sh locally (or Cell 10 below) to build the zip")
print("  3. Upload pokemage_agent.zip to the competition submission page")

---
## Cell 10 — (Optional) Build submission zip directly on Kaggle

In [ ]:
%%bash
cd /kaggle/working

ZIPFILE="/kaggle/working/pokemage_agent.zip"
SUBDIR="/kaggle/working/submission"

mkdir -p "$SUBDIR"

# Create main.py entrypoint at the top level
cat << 'EOF' > "$SUBDIR/main.py"
"""
main.py — Kaggle cabt Pokémon TCG AI Challenge submission entry point.
"""
import os
import sys

if "__file__" in globals():
    _HERE = os.path.dirname(os.path.abspath(__file__))
else:
    _HERE = "/kaggle_simulations/agent"
    if not os.path.isdir(_HERE):
        _HERE = os.path.dirname(os.path.abspath(sys.argv[0])) if (globals().get("sys") and sys.argv) else os.getcwd()

if _HERE not in sys.path:
    sys.path.insert(0, _HERE)

from agent import agent as _agent_fn

def agent(obs, config):
    return _agent_fn(obs, config)
EOF
echo "  ✓ main.py"

# Copy all supporting source files
for f in agent.py config.py card_data.py env_wrapper.py model.py train.py eval.py; do
    cp "/kaggle/working/src/$f" "$SUBDIR/$f" && echo "  ✓ $f"
done

# Bundle CSV and checkpoint
[ -f /kaggle/working/EN_Card_Data.csv ] && \
    cp /kaggle/working/EN_Card_Data.csv "$SUBDIR/" && echo "  ✓ EN_Card_Data.csv"

[ -f /kaggle/working/checkpoints/latest.pth ] && \
    cp /kaggle/working/checkpoints/latest.pth "$SUBDIR/latest.pth" && \
    echo "  ✓ latest.pth  ($(du -sh /kaggle/working/checkpoints/latest.pth | cut -f1))"

# Zip
rm -f "$ZIPFILE"
(cd "$SUBDIR" && zip -qr "$ZIPFILE" . -x '*.pyc' -x '__pycache__/*')
echo ""
echo "  ✓ pokemage_agent.zip  ($(du -sh $ZIPFILE | cut -f1))"
echo ""
echo "  Download from: Output tab → pokemage_agent.zip"
echo "  Then upload to the competition submission page."

---
## Reference: Key parameters

| Parameter | Recommended (Kaggle) | Description |
|---|---|---|
| `TOTAL_GAMES` | 5000 (one session) | More = better policy |
| `BATCH_SIZE` | 200 | Subprocess restart interval (memory leak mitigation) |
| `ALGO` | `ppo` | PPO for stable long-run training |
| `ENV` | `live` | Real cabt engine |
| `BATCH_GAMES` | 20 | Episodes per gradient update |
| `LR` | `3e-4` | Lower (1e-4) for fine-tuning a checkpoint |

## Checkpoint resume across sessions

```
Session 1: train 5,000 games → save checkpoint → create pokemage-weights dataset
Session 2: attach pokemage-weights → Cell 4 resumes → train 5,000 more games
Session 3: ...
```

Each session picks up exactly where the last one ended thanks to `game_idx` stored in the `.pth` file.
